## Memory-based CF (neighborhood-based CF)
### User-based CF (UCF)
### Item-based CF (ICF)
### Hybrid Collaborative Filtering

### Initialize

In [79]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import math


In [80]:
p1 = "../Dataset/ml-100k/u1.base"
p2 = "../Dataset/ml-100k/u1.test"

# u2 test
# p1 = "../Dataset/ml-100k/u2.base"
# p2 = "../Dataset/ml-100k/u2.test"

df1 = pd.read_csv(p1, sep='\t',
                 names=['user_id', 'item_id', 'rating', 'timestamp'])
print(df1.head())

df2 = pd.read_csv(p2,sep='\t',
                  names=['user_id','item_id','rating','timestamp'])
print(df2.head())


   user_id  item_id  rating  timestamp
0        1        1       5  874965758
1        1        2       3  876893171
2        1        3       4  878542960
3        1        4       3  876893119
4        1        5       3  889751712
   user_id  item_id  rating  timestamp
0        1        6       5  887431973
1        1       10       3  875693118
2        1       12       5  878542960
3        1       14       5  874965706
4        1       17       3  875073198


In [81]:
# Preparations
user_items = {} # {user_id:{item_id:[rating1,rating2,...]}}
for (u,i,r,ts) in df1.values:
    if u not in user_items:
        user_items[u] = {}
    user_items[u][i] = r
# print(user_items[1])
# print(user_items[1][2])

item_users = {} # {item_id:{user_id:[rating1,rating2,...]}}
for (u,i,r,ts) in df1.values:
    if i not in item_users:
        item_users[i] = {}
    item_users[i][u] = r
# print(item_users[2])
# print(item_users[2][1])

In [82]:
# Ratings
user_ratings = {} # {user_id:[rating1,rating2,...]}
item_ratings = {} # {item_id:[rating1,rating2,...]}
for (u,i,r,ts) in df1.values:
    if u not in user_ratings:
        user_ratings[u] = []
    user_ratings[u].append(r)
    if i not in item_ratings:
        item_ratings[i] = []
    item_ratings[i].append(r)
    
# print(user_ratings)
# print(user_ratings[2])
# print(user_ratings[2][0])

# print(item_ratings)
# print(item_ratings[1])
# print(item_ratings[1][1])

In [83]:
# 计算全局平均评分
# mu = df1['rating'].mean()

total = 0
count = 0
for (u,i,r,ts) in df1.values:
    total += r
    count += 1
mu = total / count

# print(mu)

# 计算用户的平均评分
# user_mean = df1.groupby('user_id')['rating'].mean()

user_mean = {}
for u,rating_list in user_ratings.items():
    user_mean[u] = sum(rating_list) / len(rating_list)

# print(user_mean)


# 计算物品的平均评分
# item_mean = df1.groupby('item_id')['rating'].mean()

item_mean = {}
for i,rating_list in item_ratings.items():
    item_mean[i] = sum(rating_list) / len(rating_list)

# print(item_mean)



### 基于用户的协同过滤(UCF)

In [84]:
## 相似度度量
## 用户u和用户w之间的皮尔逊相关系数(Pearson correlation coefficient,PCC)
S_wu = {}
users = list(user_items.keys())
# users = list(df1['user_id'].unique())
# print(users)

for idx_w in range(len(users)):
    w = users[idx_w]
    items_w = user_items[w]
    r_bar_w = user_mean[w]
    for idx_u in range(idx_w + 1,len(users)):
        u = users[idx_u]
        items_u = user_items[u]
        r_bar_u = user_mean[u]
        common_items = []
        for item in items_w:
            if item in items_u:
                common_items.append(item)
        if len(common_items) < 2:
            continue

        numerator = 0.0
        for item in common_items:
            numerator += (items_u[item] - r_bar_u) * (items_w[item] - r_bar_w)

        denom_w = 0.0
        for item in common_items:
            denom_w += pow((items_w[item] - r_bar_w),2)

        denom_u = 0.0
        for item in common_items:
            denom_u += pow(items_u[item] - r_bar_u,2)

        denominator = math.sqrt(denom_w) * math.sqrt(denom_u)
        if denominator == 0.0:
            continue

        # pcc = numerator / denominator
        # significance = len(common_items) / (len(common_items) + 50)
        # S_wu[(w,u)] = pcc * significance

        S_wu[(w,u)] = numerator / denominator 


In [85]:
# for (a,b),c in S_wu.items():
#     print(a,b)
#     print(c)
    

In [86]:
## 邻居选择
## 对于目标用户u，把与之相似度最高的K个人挑出来:
# 将S_wu按用户组织好：{u:{w:sim,...}}
user_sims = {}
for (w,u), sim in S_wu.items():
    if w not in user_sims:
        user_sims[w] = {}
    if u not in user_sims:
        user_sims[u] = {}
    user_sims[w][u] = sim
    user_sims[u][w] = sim
    # Pearson是对称的，因此两个方向都要存
# print(user_sims[1])
# print(user_sims[1][2])

In [87]:
# 选top-k

# K = 50 # 邻居数(adjustable)
# neighbors = {}

# for u in user_sims:
#     # 取走用户u的所有相似度键值对:
#     sim_pairs = list(user_sims[u].items()) #[(w1,sim1),(w2,sim2),...]

#     # 按相似度(sim)降序排序
#     for i in range(len(sim_pairs)):
#         for j in range(i+1,len(sim_pairs)):
#             if sim_pairs[i][1] < sim_pairs[j][1]:
#                 sim_pairs[i],sim_pairs[j] = sim_pairs[j],sim_pairs[i]

#     # 取走前K个:
#     top_k = []
#     for idx in range(min(K,len(sim_pairs))):
#         top_k.append(sim_pairs[idx])
#     neighbors[u] = top_k





In [88]:
# print(neighbors[1])
# print(neighbors[1][2])

In [89]:
# # 选top-k: 过滤掉负相似度
# K = 50
# neighbors = {}

# for u in user_sims:
#     sim_pairs = list(user_sims[u].items())

#     # 过滤负相似度
#     positive = []
#     for (w, sim) in sim_pairs:
#         if sim > 0:
#             positive.append((w, sim))

#     # 排序：降序
#     for i in range(len(positive)):
#         for j in range(i + 1, len(positive)):
#             if positive[i][1] < positive[j][1]:
#                 positive[i], positive[j] = positive[j], positive[i]

#     # 取前 K 个
#     top_k = []
#     for idx in range(min(K, len(positive))):
#         top_k.append(positive[idx])
#     neighbors[u] = top_k


#### Prediction Rules

In [90]:
# preds_ucf = []
# for u,i,r,ts in df2.values:
#     numerator = 0.0
#     denominator = 0.0
#     for w,sim in neighbors.get(u,[]):
#         if i in user_items.get(w,{}): # 邻居w评价过物品i吗
#             r_bar_w = user_mean[w]
#             numerator += sim * (user_items[w][i] - r_bar_w)
#             denominator += abs(sim)
#             # denominator += sim
#     if denominator == 0.0:
#         r_hat = user_mean[u]
#     else:
#         r_hat = user_mean[u] + numerator / denominator

#     if r_hat < 1:
#         r_hat = 1
#     elif r_hat > 5:
#         r_hat = 5
    
#     preds_ucf.append(r_hat)

# preds_ucf = []
# for u, i, r, ts in df2.values:
#     # 1) 先找"评过物品 i"的所有用户（排除 u 自己）
#     candidates = [w for w in user_items if i in user_items[w] and w != u]

#     # 2) 在这些用户里，按与 u 的相似度取 top-K（只留 sim > 0）
#     scored = [(w, user_sims[u].get(w, 0.0)) for w in candidates]
#     scored = [(w, s) for (w, s) in scored if s > 0]
#     scored.sort(key=lambda x: -x[1])          # 降序
#     top_k = scored[:K]

#     # 3) 加权平均（此时 top_k 里的人一定都评过 i）
#     numerator = 0.0
#     denominator = 0.0
#     for w, sim in top_k:
#         numerator += sim * (user_items[w][i] - user_mean[w])
#         denominator += abs(sim)

#     r_hat = user_mean[u] + numerator / denominator if denominator > 0 else user_mean[u]
#     preds_ucf.append(r_hat)

K = 50
preds_ucf = []
for u, i, r, ts in df2.values:
    # 1) 先找"评过物品 i"的所有用户（排除 u 自己）
    candidates = [w for w in user_items if i in user_items[w] and w != u]

    # 2) 在这些用户里，按与 u 的相似度取 top-K（只留 sim > 0）
    scored = [(w, user_sims[u].get(w, 0.0)) for w in candidates]
    scored = [(w, s) for (w, s) in scored if s > 0]
    scored.sort(key=lambda x: -x[1])          # 降序
    top_k = scored[:K]

    # 3) 加权平均（此时 top_k 里的人一定都评过 i）
    numerator = 0.0
    denominator = 0.0
    for w, sim in top_k:
        numerator += sim * (user_items[w][i] - user_mean[w])
        denominator += abs(sim)

    r_hat = user_mean[u] + numerator / denominator if denominator > 0 else user_mean[u]

    if r_hat < 1:
        r_hat = 1
    elif r_hat > 5:
        r_hat = 5

    preds_ucf.append(r_hat)




#### Metrics

In [91]:
sse = 0.0
sae = 0.0
for idx, (u, i, r, ts) in enumerate(df2.values):
    err = r - preds_ucf[idx]
    sse += err ** 2
    sae += abs(err)

print(f"UCF  RMSE = {(sse / len(preds_ucf))**0.5:.4f}")
print(f"UCF  MAE  = {sae / len(preds_ucf):.4f}")


UCF  RMSE = 0.9544
UCF  MAE  = 0.7467


### 基于物品的协同过滤(ICF)

In [92]:
## 相似度度量
## 调整后的物品k和物品j之间的余弦相似度(adjusted cosine similarity,ACS)

# ACS
S_kj = {}
items = list(item_users.keys())

for idx_k in range(len(items)):
    k = items[idx_k]
    users_k = item_users[k]        # {user_id: rating}
    for idx_j in range(idx_k + 1, len(items)):
        j = items[idx_j]
        users_j = item_users[j]

        common_users = []
        for user in users_k:
            if user in users_j:
                common_users.append(user)

        if len(common_users) < 2:
            continue

        numerator = 0.0
        for user in common_users:
            r_bar_user = user_mean[user]  
            numerator += (users_k[user] - r_bar_user) * (users_j[user] - r_bar_user)

        denom_k = 0.0
        for user in common_users:
            r_bar_user = user_mean[user]
            denom_k += (users_k[user] - r_bar_user) ** 2

        denom_j = 0.0
        for user in common_users:
            r_bar_user = user_mean[user]
            denom_j += (users_j[user] - r_bar_user) ** 2

        denominator = math.sqrt(denom_k) * math.sqrt(denom_j)
        if denominator == 0.0:
            continue
        # pcc = numerator / denominator
        # significance = len(common_users) / (len(common_users) + 50)
        # S_kj[(k,j)] = pcc * significance
        
        S_kj[(k, j)] = numerator / denominator


In [93]:
## 邻居选择
## 对于目标物品j，把与之相似度最高的K个物品挑出来:
# 将S_kj按物品组织好：{j:{k:sim,...}}
item_sims = {}
for (k,j), sim in S_kj.items():
    if k not in item_sims:
        item_sims[k] = {}
    if j not in item_sims:
        item_sims[j] = {}
    item_sims[k][j] = sim
    item_sims[j][k] = sim
    # ACS是对称的，因此两个方向都要存


In [94]:
# 选top-k

# K = 50 # 邻居数(adjustable)
# neighbors = {}

# for j in item_sims:
#     # 取走用户u的所有相似度键值对:
#     sim_pairs = list(item_sims[j].items()) #[(k1,sim1),(k2,sim2),...]

#     positive = []
#     for(k,sim) in sim_pairs:
#         if sim > 0:
#             positive.append((k,sim))

#     # 按相似度(sim)降序排序
#     for i_ in range(len(positive)):
#         for j_ in range(i_+1,len(positive)):
#             if positive[i_][1] < positive[j_][1]:
#                 positive[i_],positive[j_] = positive[j_],positive[i_]

#     # 取走前K个:
#     top_k = []
#     for idx in range(min(K,len(positive))):
#         top_k.append(positive[idx])
        
#     neighbors[j] = top_k


# print(neighbors)



#### Prediction Rules

In [95]:
# preds_icf = []
# for u, i, r, ts in df2.values:
#     # r_bar_u = user_mean.get(u,mu)
#     numerator = 0.0
#     denominator = 0.0

#     for k, sim in neighbors.get(i, []):         # i 的目标邻居物品 k
#         if u in item_users.get(k, {}):           # 用户 u 评过邻居物品 k 吗
#             numerator += sim * item_users[k][u]  # sim × r_uk
#             # numerator += sim * (item_users[k][u] - r_bar_u) # 中心化
#             denominator += abs(sim)

#     if denominator == 0.0:
#         # 没有邻居物品被 u 评过 → 退回该物品的全局均值
#         r_hat = item_mean.get(i, user_mean[u])
#     else:
#         r_hat = numerator / denominator

#     if r_hat < 1:
#         r_hat = 1
#     elif r_hat > 5:
#         r_hat = 5
        
#     preds_icf.append(r_hat)


K = 50

preds_icf = []
for u, i, r, ts in df2.values:
    # 1) 候选物品 = 用户 u 评过的所有物品（排除 i 自己）
    candidates = [k for k in item_users if u in item_users[k] and k != i]

    # 2) 在其中按与 i 的相似度取 top-K（只留 sim > 0）
    scored = [(k, item_sims.get(i,{}).get(k, 0.0)) for k in candidates]
    scored = [(k, s) for (k, s) in scored if s > 0]
    scored.sort(key=lambda x: -x[1])          # 相似度降序
    top_k = scored[:K]

    # 3) 中心化加权平均（与 UCF 对称）
    numerator = 0.0
    denominator = 0.0
    for k, sim in top_k:
        numerator   += sim * item_users[k][u]
        denominator += abs(sim)

    if denominator > 0:
        r_hat = numerator / denominator
    else:
        r_hat = item_mean.get(i, user_mean[u])   # 无邻居 -> 物品均值(或用户均值)兜底

    if r_hat < 1:
        r_hat = 1
    elif r_hat > 5:
        r_hat = 5

    preds_icf.append(r_hat)



#### Metrics

In [96]:
sse = 0.0
sae = 0.0
for idx, (u, i, r, ts) in enumerate(df2.values):
    err = r - preds_icf[idx]
    sse += err ** 2
    sae += abs(err)

print(f"ICF  RMSE = {(sse / len(preds_icf))**0.5:.4f}")
print(f"ICF  MAE  = {sae / len(preds_icf):.4f}")

ICF  RMSE = 0.9845
ICF  MAE  = 0.7753


### 混合协同过滤

In [97]:
preds_hybrid = []
lamda_ucf = 0.5
for r_ucf, r_icf in zip(preds_ucf, preds_icf):
    preds_hybrid.append(lamda_ucf * r_ucf + (1 - lamda_ucf) * r_icf)

# print(preds_hybrid)


#### Metric

In [98]:
sse = 0.0
sae = 0.0
for idx, (u, i, r, ts) in enumerate(df2.values):
    err = r - preds_hybrid[idx]
    sse += err ** 2
    sae += abs(err)

print(f"Hybrid  RMSE = {(sse / len(preds_hybrid))**0.5:.4f}")
print(f"Hybrid  MAE  = {sae / len(preds_hybrid):.4f}")

Hybrid  RMSE = 0.9526
Hybrid  MAE  = 0.7508
